# Exploration de Products — phase 1

Lecture seule des CSV sur C:, avec le .venv du projet. Grain : une ligne par product_id.

Exécuter de haut en bas depuis la racine ou notebooks/. Aucun CSV réécrit et aucun validateur appelé. Les noms source product_name_lenght et product_description_lenght sont conservés.

Le code a été exécuté avec Python pendant l'audit du 2026-10-24. Notebook fourni sans sorties sauvegardées ni lancement de noyau Jupyter. Rapport détaillé : docs/PRODUCTS_EXPLORATION.md.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
products_path = PROJECT_ROOT / "data" / "olist_products_dataset.csv"
items_path = PROJECT_ROOT / "data" / "olist_order_items_dataset.csv"
translation_path = PROJECT_ROOT / "data" / "product_category_name_translation.csv"
sources = [products_path, items_path, translation_path]
source_hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sources}

products = pd.read_csv(products_path)
raw_text = pd.read_csv(products_path, dtype="string", keep_default_na=False)
items = pd.read_csv(items_path, usecols=["order_id", "product_id"])
translation = pd.read_csv(translation_path)

report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "executable": sys.executable},
    "source_sha256": source_hashes,
    "source_size_bytes": products_path.stat().st_size,
    "shape": list(products.shape),
    "memory_deep_MiB": round(products.memory_usage(deep=True).sum() / 2**20, 3),
    "dtypes": {name: str(dtype) for name,dtype in products.dtypes.items()},
    "missing": {name: int(n) for name,n in products.isna().sum().items()},
    "missing_percent": {name: round(float(n), 4) for name,n in
                        products.isna().mean().mul(100).items()},
    "blank_or_whitespace": {name: int(raw_text[name].str.strip().eq("").sum())
                            for name in products.columns},
    "whitespace_only_nonempty": {name: int((raw_text[name].ne("")
        & raw_text[name].str.strip().eq("")).sum()) for name in products.columns},
    "exact_duplicate_rows_excess": int(products.duplicated().sum()),
    "duplicate_product_id_rows": int(products.product_id.duplicated(keep=False).sum()),
    "distinct_product_ids": int(products.product_id.nunique()),
    "product_id_format_mismatch": int((~raw_text.product_id.str.fullmatch(
        r"[0-9a-f]{32}", na=False)).sum()),
}
print(json.dumps(report, indent=2, ensure_ascii=False))


## Valeurs numériques et manques

Distinguer les valeurs absentes (NaN après lecture CSV), non convertibles, infinies, négatives et égales à zéro. Un type float64 peut résulter de la présence de nulls dans une colonne de compteurs.

Les contrôles n'attribuent aucun statut valid/review. Les manques et les zéros ne sont pas automatiquement invalides. Les statistiques excluent les valeurs absentes.


In [ ]:
numeric_columns = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
report["numeric_statistics"] = products[numeric_columns].describe(
    percentiles=[.25, .5, .75, .95, .99]).to_dict()
report["numeric_checks"] = {}
for column in numeric_columns:
    values = pd.to_numeric(products[column], errors="coerce")
    finite = values.notna() & np.isfinite(values)
    report["numeric_checks"][column] = {
        "missing": int(products[column].isna().sum()),
        "non_numeric_non_null": int((products[column].notna() & values.isna()).sum()),
        "non_finite_non_null": int((values.notna() & ~np.isfinite(values)).sum()),
        "negative": int(values.lt(0).sum()),
        "zero": int(values.eq(0).sum()),
        "non_integer_finite": int((finite & values.where(finite).mod(1).ne(0)).sum()),
    }
metadata_columns = ["product_category_name", "product_name_lenght",
                    "product_description_lenght", "product_photos_qty"]
physical_columns = ["product_weight_g", "product_length_cm",
                    "product_height_cm", "product_width_cm"]
metadata_missing = products[metadata_columns].isna()
physical_missing = products[physical_columns].isna()
report["missing_patterns"] = {
    "any_missing_rows": int(products.isna().any(axis=1).sum()),
    "all_metadata_missing_rows": int(metadata_missing.all(axis=1).sum()),
    "any_metadata_missing_rows": int(metadata_missing.any(axis=1).sum()),
    "all_physical_missing_rows": int(physical_missing.all(axis=1).sum()),
    "any_physical_missing_rows": int(physical_missing.any(axis=1).sum()),
    "both_groups_missing_rows": int((metadata_missing.any(axis=1)
                                    & physical_missing.any(axis=1)).sum()),
}
print(json.dumps({"numeric_statistics": report["numeric_statistics"],
                  "numeric_checks": report["numeric_checks"],
                  "missing_patterns": report["missing_patterns"]}, indent=2))


## Catégories

La distribution compte les lignes produit, pas les articles vendus ni un revenu. Une catégorie absente et une catégorie présente mais non traduite sont deux situations distinctes. Aucun nom de catégorie n'est inventé.


In [ ]:
category_counts = products.product_category_name.value_counts(dropna=True)
report["categories"] = {
    "distinct_non_null": int(category_counts.size),
    "missing_product_rows": int(products.product_category_name.isna().sum()),
    "distribution_product_rows": {str(name): int(n) for name,n in category_counts.items()},
    "source_surrounding_whitespace": int(raw_text.product_category_name.ne(
        raw_text.product_category_name.str.strip()).sum()),
}
print(json.dumps(report["categories"], ensure_ascii=False, indent=2))
# These counts measure distinct product rows, not sales, item volume or revenue.
# The source uses the spelling "lenght"; do not rename raw columns.


## Références et traduction

Jointures many_to_one pour éviter une multiplication des lignes. Vérifier les produits d'Order Items, ceux qui ne sont pas utilisés et la couverture de traduction. Un produit non commandé ne serait pas automatiquement invalide dans un futur catalogue.

Ne pas imputer une traduction aux catégories inconnues du fichier de correspondance. Les valeurs manquantes restent inchangées.


In [ ]:
# Enforce many-to-one joins to avoid hiding duplicate reference keys.
items_with_products = items.merge(
    products, on="product_id", how="left", validate="many_to_one", indicator=True)
unused_products = products.loc[~products.product_id.isin(items.product_id)]

translation_join = products.merge(
    translation, on="product_category_name", how="left",
    validate="many_to_one", indicator=True)
missing_category = products.product_category_name.isna()
unknown_categories = (
    ~missing_category
    & ~products.product_category_name.isin(translation.product_category_name)
)
unknown_category_names = sorted(
    products.loc[unknown_categories, "product_category_name"].unique().tolist())
report["relations"] = {
    "item_rows": int(len(items)),
    "distinct_product_ids_in_items": int(items.product_id.nunique()),
    "joined_item_rows": int(len(items_with_products)),
    "item_rows_unknown_product": int(items_with_products["_merge"].eq("left_only").sum()),
    "unknown_product_ids_in_items": int(items.loc[
        ~items.product_id.isin(products.product_id), "product_id"].nunique()),
    "unused_product_rows": int(len(unused_products)),
    "item_rows_with_missing_category": int(items_with_products.product_category_name.isna().sum()),
    "item_rows_with_missing_physical_data": int(
        items_with_products[physical_columns].isna().any(axis=1).sum()),
}
report["translation"] = {
    "shape": list(translation.shape),
    "columns": translation.columns.tolist(),
    "missing": {name: int(n) for name,n in translation.isna().sum().items()},
    "blank_or_whitespace": {name: int(translation[name].astype("string").str.strip().eq("").sum())
                            for name in translation.columns},
    "exact_duplicate_rows_excess": int(translation.duplicated().sum()),
    "duplicate_category_rows": int(translation.product_category_name.duplicated(keep=False).sum()),
    "joined_product_rows": int(len(translation_join)),
    "untranslated_non_null_product_rows": int(unknown_categories.sum()),
    "untranslated_non_null_category_names": unknown_category_names,
    "untranslated_non_null_counts": {str(name): int(n) for name,n in
        products.loc[unknown_categories,"product_category_name"].value_counts().items()},
    "item_rows_with_untranslated_non_null_category": int(
        items_with_products.product_category_name.isin(unknown_category_names).sum()),
    "translation_categories_unused_in_products": sorted(
        set(translation.product_category_name.dropna()) -
        set(products.product_category_name.dropna())),
}
assert source_hashes == {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                         for p in sources}, "Source changed during analysis"
print(json.dumps({"relations": report["relations"],
                  "translation": report["translation"]}, ensure_ascii=False, indent=2))
# Missing translations do not prove the Portuguese category itself is invalid.
# Missing metadata and zero measurements remain observations, not quality flags.



## Propositions avant le validateur
Schéma et clé produit obligatoires ; nombres présents convertibles, finis et non négatifs ; compteurs/longueurs descriptives entiers. Catégorie, métadonnées et mesures absentes : politique proposée permissive avec indicateurs de complétude, à faire valider.
Poids nul, traduction absente et extrêmes : indicateurs métier, sans correction automatique. Ne pas prendre les maxima observés comme bornes. Ne pas renommer les colonnes source. Aucun products_validator.py créé.
